# Notebook 03: List Comprehensions and Machine Learning Activation Functions

This notebook examines Python list comprehensions and their application in constructing neural network activation functions:
1. **Comprehension Syntax & Semantics:** Mapping vs filtering vs ternary branching.
2. **Common Syntax Pitfalls:** When and why `if` goes at the end vs before `for`.
3. **Sigmoid Activation & Derivative:** $\sigma(x) = rac{1}{1 + e^{-x}}$, $\sigma'(x) = \sigma(x)(1 - \sigma(x))$.
4. **ReLU & Leaky ReLU:** Rectified activations and subgradient implementations.
5. **Softmax Normalization:** Vector transformation with numerical stability tricks.


In [ ]:
import csv
import math
from pathlib import Path

# Load test activation inputs
CSV_PATH = Path("../data/activation_inputs.csv")
records = []
with CSV_PATH.open("r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        records.append(
            {
                "sample_id": row["sample_id"],
                "input_value": float(row["input_value"]),
                "expected_relu": float(row["expected_relu"]),
                "expected_sigmoid": float(row["expected_sigmoid"]),
                "expected_leaky_relu": float(row["expected_leaky_relu"]),
            }
        )

print(f"Loaded {len(records)} activation test records.")

## 1. List Comprehension Mechanics: Mapping, Filtering, and Branching

- **Mapping:** `[f(x) for x in data]`
- **Filtering:** `[x for x in data if condition(x)]`
- **Branching (Ternary):** `[a if condition(x) else b for x in data]`


In [ ]:
data = [1, 5, -4, 3, -2]

# Quiz 1: Branching with ternary expression (replacement)
clipped_pos = [x if x > 0 else 0 for x in data]
assert clipped_pos == [1, 5, 0, 3, 0]

# Quiz 3: Filtering (length changes)
filtered_pos = [x for x in data if x > 0]
assert filtered_pos == [1, 5, 3]

# Verifying syntax error mechanics:
# 'if' without 'else' before 'for' is invalid syntax in Python:
# [x if x > 0 for x in data] -> SyntaxError

# 'else' in post-filter clause is invalid syntax in Python:
# [x for x in data if x > 0 else 0] -> SyntaxError

print("Comprehension syntax mechanics confirmed!")

## 2. Neural Network Activations: Sigmoid

$$\sigma(x) = rac{1}{1 + e^{-x}}, \quad \sigma'(x) = \sigma(x)(1 - \sigma(x))$$


In [ ]:
def sigmoid_scalar(x: float) -> float:
    """Compute logistic sigmoid for a single real scalar with stability clamp."""
    if x < -700:
        return 0.0
    if x > 700:
        return 1.0
    return 1.0 / (1.0 + math.exp(-x))


def sigmoid_derivative_scalar(x: float) -> float:
    """Compute analytical derivative of sigmoid function."""
    s = sigmoid_scalar(x)
    return s * (1.0 - s)


def sigmoid_list(values: list[float]) -> list[float]:
    """Vectorize sigmoid over a Python list using comprehension."""
    return [sigmoid_scalar(x) for x in values]


def sigmoid_derivative_list(values: list[float]) -> list[float]:
    """Vectorize sigmoid derivative over a Python list using comprehension."""
    return [sigmoid_derivative_scalar(x) for x in values]


# Validate with CSV benchmarks
test_inputs = [r["input_value"] for r in records]
computed_sigmoid = sigmoid_list(test_inputs)

for r, computed in zip(records, computed_sigmoid, strict=True):
    assert math.isclose(computed, r["expected_sigmoid"], rel_tol=1e-5)

print("Sigmoid activation implementation passes all ground truth tests!")

## 3. Neural Network Activations: ReLU & Leaky ReLU

$$\text{ReLU}(x) = \max(0, x), \quad \text{ReLU}'(x) = \begin{cases} 0 & x \le 0 \\ 1 & x > 0 \end{cases}$$

$$\text{LeakyReLU}(x, \alpha) = \begin{cases} \alpha x & x \le 0 \\ x & x > 0 \end{cases}$$


In [ ]:
def relu_list(values: list[float]) -> list[float]:
    """Compute ReLU activation element-wise using comprehension with branching."""
    return [x if x > 0 else 0.0 for x in values]


def relu_derivative_list(values: list[float]) -> list[float]:
    """Compute subgradient derivative of ReLU element-wise."""
    return [1.0 if x > 0 else 0.0 for x in values]


def leaky_relu_list(values: list[float], alpha: float = 0.01) -> list[float]:
    """Compute Leaky ReLU activation element-wise."""
    return [x if x > 0 else alpha * x for x in values]


computed_relu = relu_list(test_inputs)
computed_leaky = leaky_relu_list(test_inputs, alpha=0.01)

for r, cr, cl in zip(records, computed_relu, computed_leaky, strict=True):
    assert math.isclose(cr, r["expected_relu"], abs_tol=1e-7)
    assert math.isclose(cl, r["expected_leaky_relu"], abs_tol=1e-7)

print("ReLU and Leaky ReLU activations pass all benchmarks!")

## 4. Softmax Probability Normalization

$$\text{Softmax}(z_i) = \frac{e^{z_i - \max(z)}}{\sum_{j} e^{z_j - \max(z)}}$$


In [ ]:
def softmax_list(logits: list[float]) -> list[float]:
    """Compute numerically stable softmax probability distribution."""
    max_logit = max(logits)
    exp_shifted = [math.exp(z - max_logit) for z in logits]
    total_exp = sum(exp_shifted)
    return [e_val / total_exp for e_val in exp_shifted]


logits = [2.0, 1.0, 0.1]
probs = softmax_list(logits)

assert math.isclose(sum(probs), 1.0, rel_tol=1e-7)
assert probs[0] > probs[1] > probs[2]
print(f"Logits: {logits}")
print(f"Softmax probabilities: {[round(p, 4) for p in probs]}")
print("All activation functions tested and validated!")